# Evaluating the Warehouse Agent with Strands Evals & AgentCore Evaluations

In Lab 7 you deployed a warehouse operations agent to Amazon Bedrock AgentCore Runtime using the A2A protocol. But an answer that *looks* right is not proof the agent is good. Before you trust it with real procurement decisions, you need to measure its quality in a repeatable way.

This lab builds an evaluation suite and grades the agent in **two stages**. Stage 1 runs the agent locally with [Strands Evals](https://pypi.org/project/strands-agents-evals/) for fast feedback while you iterate. Stage 2 deploys the Stage-1 fix as a new A2A-protocol runtime and grades it with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/), reading the production OTEL traces.

> **No other notebook needs to be running.** This lab loads Cognito credentials from the environment variables set in Lab 7. It does require **Lab 7 to have been completed**: the setup section below stops immediately if Cognito env vars are missing or AWS credentials are not configured.

If you have not set up your AI Core credentials yet, follow notebook 00-load-sap-ai-core-credentials first.

## Prerequisites

1. **Completed Lab 7** (A2A warehouse agent deployed -- `CLIENT_ID`, `USER_POOL_ID`, `DISCOVERY_URL`, `COGNITO_DOMAIN`, and `OAUTH_SCOPE` must be set in your environment).
2. SAP AI Core credentials in `~/.aicore/config.json` (run Lab 00 if you have not).
3. An SAP S/4HANA Public Cloud API key, either in `.env` or entered when prompted.
4. AWS credentials with AgentCore Evaluation permissions.
5. **CloudWatch Transaction Search enabled** in this account/region (required for Stage 2). AgentCore Runtime emits OpenTelemetry spans automatically: the starter toolkit builds the container with `aws-opentelemetry-distro` and runs it under `opentelemetry-instrument`, so you do **not** configure spans in the agent code. But those spans only become queryable traces once **Transaction Search** is turned on at the account level; without it, `EvaluationClient.run()` finds no trace to grade. The Stage 2 preflight cell checks this for you and prints setup instructions if it is off. See [Transaction Search setup](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html).

The setup section below validates AWS credentials and Cognito env vars up front, so items 1-4 are needed before Stage 1 runs. Transaction Search (item 5) is only required for Stage 2.

## 1. Setup: imports, configuration, and the local agent

The next few cells import dependencies, check your credentials, initialize the `LiteLLMModel` (routing through SAP GenAI Hub via the `sap/` prefix), load Cognito configuration from environment variables, and build the local warehouse agent used in Stage 1.


In [ ]:
from strands.models.litellm import LiteLLMModel
import warnings
# Suppress a known LiteLLM cosmetic warning: its logging worker shuts down a background
# thread before an async success handler coroutine is awaited. LLM calls still complete
# successfully — this is noise from LiteLLM's internal queue teardown, not a real error.
warnings.filterwarnings(
    "ignore",
    message="coroutine.*was never awaited",
    category=RuntimeWarning,
    module="litellm",
)
# "Task was destroyed but it is pending!" is printed by asyncio's internal logger
# when LiteLLM's LoggingWorker task is garbage-collected without being cancelled.
# Filter it at the logging level since it bypasses the warnings module entirely.
import logging
logging.getLogger("asyncio").addFilter(
    type("_LiteLLMTaskFilter", (logging.Filter,), {
        "filter": lambda self, r: "LoggingWorker" not in r.getMessage()
    })()
)

import os
import json
import time
import uuid
from datetime import timedelta
from collections import defaultdict

import boto3
from dotenv import load_dotenv
import getpass

load_dotenv()

# Prompt for the SAP API key if it isn't already set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")


In [ ]:
# Display helpers: render results as real Markdown (rich HTML tables and prose) instead of
# fixed-width text, the same way Lab 7 renders the agent's answer with IPython's Markdown.
from IPython.display import Markdown, display


def show_md(text, title=None):
    """Render `text` as Markdown in the notebook. With `title`, prepend an H3 heading.

    Use this for agent answers and free-form diagnoses so headings, bold, and lists in the
    text render as HTML (exactly Lab 7's `display(Markdown(...))` pattern), not gray monospace.
    """
    display(Markdown(f"### {title}\n\n{text}" if title else text))


def md_table(headers, rows, title=None, note=None):
    """Render a scorecard as a Markdown table (columns auto-size, renders as HTML).

    Replaces the hand-aligned `f"{v:>14}"` / `"=" * 108` tables. `headers` is a list of column
    labels; `rows` is a list of row lists (cells are stringified). Optional `title` (H3 heading)
    and `note` (a paragraph rendered below the table).
    """
    lines = []
    if title:
        lines.append(f"### {title}\n")
    lines.append("| " + " | ".join(str(h) for h in headers) + " |")
    lines.append("| " + " | ".join("---" for _ in headers) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    if note:
        lines.append(f"\n{note}")
    display(Markdown("\n".join(lines)))


In [ ]:
# Validate required configuration before proceeding
_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json. Run notebook 00 first.")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set. Check your .env file.")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
# TODO: Choose your model. The sap/ prefix routes through SAP GenAI Hub via LiteLLM.
# Options: "sap/anthropic--claude-4.5-sonnet", "sap/amazon--nova-pro", "sap/amazon--nova-lite"
model = LiteLLMModel(model_id="sap/anthropic--claude-4.5-sonnet")

# Load Cognito + region configuration written by Lab 7. Stage 2 deploys a new A2A runtime
# secured with the same Cognito JWT authorizer and invokes it with a Cognito bearer token.
REGION        = os.environ.get("REGION") or boto3.session.Session().region_name or "us-east-1"
CLIENT_ID     = os.environ.get("CLIENT_ID", "")
USER_POOL_ID  = os.environ.get("USER_POOL_ID", "")
DISCOVERY_URL = os.environ.get("DISCOVERY_URL", "")
COGNITO_DOMAIN = os.environ.get("COGNITO_DOMAIN", "")
OAUTH_SCOPE   = os.environ.get("OAUTH_SCOPE", "")

missing = [k for k, v in {
    "CLIENT_ID": CLIENT_ID, "USER_POOL_ID": USER_POOL_ID,
    "DISCOVERY_URL": DISCOVERY_URL, "COGNITO_DOMAIN": COGNITO_DOMAIN,
    "OAUTH_SCOPE": OAUTH_SCOPE,
}.items() if not v]
if missing:
    raise SystemExit(
        f"Missing Cognito env vars: {missing}\n"
        "Run Lab 7 (07-deploy-warehouse-a2a-agent-to-agentcore.ipynb) steps 5-6c first."
    )

# AGENT_ARN / AGENT_ID are populated by the Stage 2 deploy cell. Empty here so Stage 1
# cells run without blocking on Stage 2.
AGENT_ARN = os.environ.get("LAB9_AGENT_ARN", "")
AGENT_ID  = os.environ.get("LAB9_AGENT_ID", "")

print(f"Region        : {REGION}")
print(f"Client ID     : {CLIENT_ID}")
print(f"Discovery URL : {DISCOVERY_URL}")
print(f"Cognito scope : {OAUTH_SCOPE}")
print(f"Model         : {model.model_id} via SAP GenAI Hub (LiteLLM)")

### The local agent

Stage 1 grades a local copy of the same warehouse agent you built in Lab 6 and deployed in Lab 7. It is a two-agent design: a **selector sub-agent** (`SelectorAPIAgentAsATool`) reads the OpenAPI specs under `assets/knowledgebase` and picks which SAP API fits the question, then the **warehouse agent** calls that selector and the `odata_caller` tool to fetch inventory data and answer. For the architecture diagram, please refer back to Lab 6. 

We also already prepare everything for an improved agent version that we will test against later. This version was created based on the feedback, when evaluting our baseline agent. 

In [ ]:
# Import the two Lab 6/7 warehouse-agent architectures from util/warehouse_agent.py:
#  - build_warehouse_agent: the BASELINE (selector sub-agent + odata_caller). It rediscovers the
#    schema at runtime via the selector + $metadata, and hand-builds $filter/$select. Stage 1
#    grades this as the naive baseline.
#  - build_improved_warehouse_agent: the eval-driven fix, based on the first round of evals. A deep `get_warehouse_stock` tool encodes
#    the entity, fields, warehouse ID, and server-side query, so a stock question is answered in one
#    call. The selector + odata_caller stay wired in as a fallback for off-path queries (a router).
# See util/warehouse_agent.py for the finding-by-finding mapping from the trajectory judge to the fix.
from util.warehouse_agent import (
    WAREHOUSE_SYSTEM_PROMPT,
    WAREHOUSE_CAPACITY,
    build_warehouse_agent,
    build_improved_warehouse_agent,
)


def create_warehouse_agent():
    """Build the local BASELINE warehouse agent (selector + odata_caller) for Stage 1."""
    return build_warehouse_agent(model, system_prompt=WAREHOUSE_SYSTEM_PROMPT)


def create_improved_warehouse_agent():
    """Build the local IMPROVED warehouse agent (deep get_warehouse_stock tool + fallback)."""
    return build_improved_warehouse_agent(model)


# Smoke-test that both agents construct.
_ = create_warehouse_agent()
_ = create_improved_warehouse_agent()
print("Local warehouse agents (baseline + improved) ready for Stage 1 evaluation.")


## How we evaluate: two stages

The two stages answer two different questions:

1. **Stage 1, local, with [Strands Evals](https://pypi.org/project/strands-agents-evals/): "is the logic good?"** Run the agent in this notebook and grade it in process. Fast and no deployment needed. This is the pre-deploy gate you iterate and fix against before you ship.
2. **Stage 2, deployed, with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/): "did the fix ship, and how does the real thing behave under production infrastructure?"** We ship Stage 1's fix to the live AgentCore Runtime, then grade it from the production OpenTelemetry (OTEL) traces it writes to CloudWatch. That means a **parity** check (did quality hold?) plus **prod-only signals** a local run cannot produce: latency and cold-start. This is also what you wire into ongoing production monitoring.

Grade locally as you build, then ship the fix and verify it in production.

<div style="text-align:center">
    <img src="assets/20260731-two-stage-agent-eval.png" width="65%" />
</div>

## Understanding grader types

Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) groups graders into three families. Most real evaluations **combine** them.

| Grader family | How it works | In this lab |
|---|---|---|
| **Code-based** | Deterministic checks: string match, counting, static analysis | `ToolCalled`, `ToolCallBudget` (local) |
| **Model-based (LLM as judge)** | An LLM scores the output against a rubric | `OutputEvaluator`, `TrajectoryEvaluator`, `ToolSelectionAccuracy` (local); `Correctness`, `Helpfulness`, `GoalSuccessRate`, `ToolSelectionAccuracy`, and the custom `WarehouseOperationalQuality` (deployed) |
| **Human** | Expert review and spot checks | Not shown here; used in practice to calibrate the LLM judges and curate the golden dataset |

Rule of thumb: use code-based graders wherever you can (cheapest and most reliable), reach for LLM judges when you need nuance, and use humans to calibrate those judges.

**There is a second question to ask about every grader: *what* is it grading?**

- **Outcome** grades the final answer. *Did the agent get it right?* Examples: `OutputEvaluator` (local), `Correctness` and `WarehouseOperationalQuality` (deployed).
- **Trajectory** grades the path taken. *Did it use the right tools, efficiently?* Examples: `ToolCalled`, `ToolCallBudget`, `TrajectoryEvaluator`, `ToolSelectionAccuracy`.

The two trajectory-efficiency graders are deliberately paired: the code-based `ToolCallBudget` is a **deterministic guardrail** (it fails the moment an agent exceeds its per-scenario tool-call budget, and gives the same verdict every run), while the LLM `TrajectoryEvaluator` supplies the **qualitative "why"** (which specific calls were wasteful). This is exactly why the "correct-but-inefficient" blind spot below is caught reproducibly rather than left to a judge's score that can drift run to run.


## Evaluation scenarios

These scenarios are **shared by both stages**: graded locally in Stage 1 and on the deployed runtime in Stage 2. Each scenario carries four things:

- **prompt**: the user query the agent must answer.
- **expected_response**: what a good answer looks like, used by the outcome graders (`OutputEvaluator` locally, `Correctness` and the custom evaluator when deployed).
- **expected_trajectory**: the tool calls we expect, consumed by the **local Stage 1** trajectory graders (`ToolSelectionAccuracyEvaluator`, `TrajectoryEvaluator`). Stage 2's built-in `ToolSelectionAccuracy` needs no ground truth (see the Stage 2 evaluator table), so this field is not required there.
- **assertions**: the goal conditions, used by `GoalSuccessRate` in Stage 2.

The evaluation scenarios will be our golden dataset for this lab. A golden dataset is a curated collection of test cases with known-correct answers that you can use as ground truth to measure how well an agent perform. Ideally, you should start curating a golden dataset early on in the agent development process to make sure iterative changes will lead to improvements. In this lab we use only a subset of 4 evaluation scenarios to illustrate, but a in a real-case scenario you would want more scenarios that cover all important cases ranging from 20 to 100+. 

> **Ground truth reflects the live SAP sandbox.** The `low-stock-items` and `inventory-overview` scenarios pin specific products and quantities in their `expected_response` (e.g. `MZ-TG-AS-H100`, `SG424`, `RM124`, and the 480-unit total for the automation products) so the outcome judges actually fail a wrong number, not just a missing product. Those values were current in the shared SAP S/4HANA sandbox when these scenarios were written, and can drift. If the outcome judges (`OutputEvaluator` / `Correctness`) start penalizing correct answers, re-run the agent once, read off the current products and totals, and update the `expected_response` below — that re-baselining step is itself part of maintaining a golden dataset.

In [ ]:
# Each scenario carries a `tool_call_budget`: the max number of tool calls a good agent should
# need to answer it. Every scenario here is a single, well-formed query, so the budget is 2 (one
# data call, plus one call of headroom). The baseline blows past it (selector + $metadata rediscovery
# + trial-and-error $filter); the improved deep-tool agent answers in one call. The code-based
# ToolCallBudget grader (next section) turns this into a deterministic pass/fail efficiency check.
# TODO: set a per-scenario budget that matches how many calls YOUR agent legitimately needs.
evaluation_scenarios = [
    {
        "name": "inventory-check-single",
        "prompt": "What is the current stock level for WM-AN02 Control Units?",
        "expected_response": (
            "The response should contain the specific stock quantity for WM-AN02 Control Units "
            "retrieved from the SAP warehouse API, with the product correctly identified as "
            "Control Units. The number should come from actual API data, not be invented."
        ),
        "expected_trajectory": ["odata_caller"],
        "tool_call_budget": 2,
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent correctly identified WM-AN02 as Control Units. "
            "Agent reported a specific numeric stock quantity from the API."
        ),
    },
    {
        "name": "inventory-overview",
        "prompt": "Give me a complete overview of all products currently in the warehouse.",
        "expected_response": (
            "The response should list all warehouse products (WM-AN01 Advanced Sensors, "
            "WM-AN02 Control Units, WM-AN03 Power Modules, WM-AN04 Communication Devices) "
            "with their current stock quantities from the API, presented in a structured format. "
            "Quantities must be PER-PRODUCT totals summed across all storage bins, not single-bin "
            "fragments. With the current data the four automation products each total 480 PC; a "
            "response that reports a different total for them (e.g. 528) is wrong."
        ),
        "expected_trajectory": ["odata_caller"],
        "tool_call_budget": 2,
        "assertions": (
            "Agent queried warehouse stock data via OData. "
            "Agent listed multiple products with stock quantities. "
            "Agent presented results in a structured, readable format."
        ),
    },
    {
        "name": "fulfillment-feasibility",
        "prompt": "Can we fulfill an order for 200 units of WM-AN02 Control Units?",
        "expected_response": (
            "The response should check current WM-AN02 stock from the API, compare it against "
            "the requested 200 units, and provide a clear yes/no fulfillment recommendation "
            "with the actual available quantity."
        ),
        "expected_trajectory": ["odata_caller"],
        "tool_call_budget": 2,
        "assertions": (
            "Agent queried current WM-AN02 stock via OData. "
            "Agent compared available quantity against the 200-unit request. "
            "Agent provided a clear yes/no fulfillment answer with supporting data."
        ),
    },
    {
        "name": "low-stock-items",
        "prompt": "What are the items with low stock?",
        "expected_response": (
            "The response should identify which products are low on stock based on PER-PRODUCT "
            "total quantities (summed across all storage bins) from the SAP warehouse API, "
            "compared against the reorder threshold of 20% of the 500-unit capacity (under 100 "
            "units). With the current data the low products are MZ-TG-AS-H100 (~4 units) and "
            "SG424 (~5 units). High-stock products such as RM124 (~1,772 units) must NOT be "
            "flagged as low: a product's total, not any single low bin, decides. The agent should "
            "reach the answer efficiently, ideally a single query, not repeated schema rediscovery."
        ),
        "expected_trajectory": ["odata_caller"],
        "tool_call_budget": 2,
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent judged low stock on per-product totals (summed across bins), not single bins. "
            "Agent correctly flagged the genuinely low products and did not flag high-stock ones. "
            "Agent reached the answer without redundant, repeated OData calls."
        ),
    },
]

print(f"Evaluation scenarios defined: {len(evaluation_scenarios)}")
for s in evaluation_scenarios:
    print(f"  - {s['name']}: {s['prompt']}  (budget: {s['tool_call_budget']} tool calls)")

## Stage 1: Local evaluation with Strands Evals

Run the agent right here in the notebook and grade it in process. This is the fast gate you iterate against before deploying. **[Strands Evals](https://pypi.org/project/strands-agents-evals/)** (already a project dependency) gives us graders from both families:

- **Code-based (deterministic):** `ToolCalled` checks that the agent used its tool at all.
- **LLM as judge:** `OutputEvaluator` scores whether the final answer is correct (task success), `TrajectoryEvaluator` scores the tool-call path against a rubric, and `ToolSelectionAccuracyEvaluator` judges whether each individual call was justified. All three route through the **same `LiteLLMModel`** the agent uses, so no separate Bedrock access is needed.

**Where the trajectory comes from.** Each scenario runs once against the local agent. We read its tool usage off the result with the SDK's `tools_use_extractor` ([docs](https://strandsagents.com/docs/user-guide/evals-sdk/quickstart/)):

```python
from strands_evals.extractors import tools_use_extractor
calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
# -> [{"name": "odata_caller", "input": {...}, "tool_result": "...", "is_error": False}, ...]
```

We keep two views of that run: a flat list of tool names for `ToolCalled`, and a detailed list (each call's OData arguments plus an `is_error` flag) for the trajectory judge, so it can name the *specific* redundant or failed calls instead of just counting them. This is what lets a scenario **pass on task success but score low on trajectory**: the right answer reached the wrong way or inefficiently.


In [ ]:
# Code-based graders with Strands Evals: deterministic, no LLM. This cell also defines the
# reusable `evaluate_agent` helper the "fix and re-test" cell below calls a second time.
from strands_evals.evaluators import Evaluator, ToolCalled
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import EvaluationData, EvaluationOutput


class ToolCallBudget(Evaluator):
    """Deterministic trajectory-efficiency grader: pass iff total tool calls <= budget.

    This is the code-based guardrail for the "correct-but-inefficient" blind spot. Where the
    LLM `TrajectoryEvaluator` (next section) explains *why* a path was wasteful but can score the
    same trajectory differently run to run, this grader is a hard, reproducible pass/fail: a
    per-scenario budget of N tool calls, and any trajectory over it fails with the same verdict
    every time. It reads the exact same flat tool-name trajectory `ToolCalled` uses (one entry per
    tool call, from the SDK's `tools_use_extractor`), so its count matches the "Calls" column and
    the SDK's own `tool_metrics` call count.
    """

    def __init__(self, budget: int):
        super().__init__()
        self.budget = budget

    def evaluate(self, evaluation_case):
        trajectory = evaluation_case.actual_trajectory
        if not isinstance(trajectory, list):
            return [EvaluationOutput(score=0.0, test_pass=False,
                                     reason="no tool-call trajectory to score")]
        n_calls = len(trajectory)
        within = n_calls <= self.budget
        return [EvaluationOutput(
            score=1.0 if within else 0.0,
            test_pass=within,
            label="within_budget" if within else "over_budget",
            reason=(f"{n_calls} tool call(s) against a budget of {self.budget} "
                    f"({'within' if within else 'OVER'} budget)"),
        )]


# The BASELINE agent answers via the generic odata_caller; the IMPROVED agent answers via the deep
# get_warehouse_stock tool. evaluate_agent takes the primary tool name so it counts and grades the
# right one for whichever architecture is under test (the whole before/after point is that this
# tool changes). TODO: change if your agent uses a different primary tool.
TOOL_NAME = "odata_caller"
IMPROVED_TOOL_NAME = "get_warehouse_stock"

# OData argument keys worth surfacing to the trajectory judge: these reveal waste
# (repeated $metadata rediscovery, trial-and-error $filter guessing).
_ODATA_KEYS_OF_INTEREST = ("endpoint", "operation", "$filter", "$orderby", "$select", "$top")

# Captured once so the trajectory judge knows what each tool does (see the LLM-judge cell).
TOOL_DESCRIPTIONS = {}


def _summarize_tool_call(call: dict) -> dict:
    """Reduce one extracted tool-call record to the fields that show what it did.
    We surface the endpoint and OData query params (so the judge sees $filter/$orderby
    directly) and keep is_error, since a failed call is the tell-tale of trial-and-error
    $filter guessing. For the deep get_warehouse_stock tool we surface its typed args instead,
    so the judge can see the whole query was expressed in a single well-formed call.
    """
    tool_input = call.get("input") or {}
    odata_params = tool_input.get("odata_params") or {}
    flat = {**tool_input, **odata_params}
    summary = {"name": call.get("name")}
    for key in (*_ODATA_KEYS_OF_INTEREST, "product", "low_stock_only"):
        if flat.get(key) is not None and flat.get(key) != "":
            summary[key] = flat[key]
    if call.get("is_error"):
        summary["is_error"] = True
    return summary


def run_local_agent_trajectory(prompt: str, make_agent):
    """Run one scenario against a local agent; return (output, names, detailed, tokens).
    - names: flat tool-name list, for `ToolCalled` (exact-match membership) and `ToolCallBudget`.
    - detailed: ordered {name, endpoint, $filter, is_error, ...} dicts, for the LLM judge.
    - tokens: total tokens the agent consumed (informational efficiency signal).
    Both trajectory views come from the SDK's `tools_use_extractor`.
    """
    global TOOL_DESCRIPTIONS
    agent = make_agent()
    result = agent(prompt)
    output_text = str(result.message)

    if not TOOL_DESCRIPTIONS:
        TOOL_DESCRIPTIONS = tools_use_extractor.extract_tools_description(agent)

    calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    names = [c.get("name") for c in calls]
    detailed = [_summarize_tool_call(c) for c in calls]
    tokens = (result.metrics.accumulated_usage or {}).get("totalTokens")
    return output_text, names, detailed, tokens


def evaluate_agent(make_agent, scenarios, tool_name=TOOL_NAME):
    """Run every scenario against `make_agent()` once and grade with the code-based graders.
    Two deterministic checks run here: `ToolCalled` (was the primary data tool used at all?) and
    `ToolCallBudget` (did the agent stay within the scenario's tool-call budget?). `tool_name` is
    the primary data tool for the architecture under test (odata_caller for the baseline,
    get_warehouse_stock for the improved agent); the expected trajectory is set to that tool so the
    LLM trajectory graders compare against the right target. Returns a per-scenario cache reused by
    the LLM-judge cell and the before/after comparison, so we invoke the agent once per scenario.
    """
    runs = {}
    for scenario in scenarios:
        name = scenario["name"]
        budget = scenario["tool_call_budget"]
        output_text, names, detailed, tokens = run_local_agent_trajectory(scenario["prompt"], make_agent)
        expected_trajectory = [tool_name]

        # name_case carries expected_output so the OutputEvaluator (task-success judge) can
        # compare the answer against what a good response should contain.
        name_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            expected_output=scenario["expected_response"],
            actual_trajectory=names, expected_trajectory=expected_trajectory,
        )
        detail_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            actual_trajectory=detailed, expected_trajectory=expected_trajectory,
        )

        tool_called = ToolCalled(tool_name).evaluate(name_case)[0]
        budget_result = ToolCallBudget(budget).evaluate(name_case)[0]
        n_calls = len(names)
        n_primary = names.count(tool_name)
        runs[name] = {
            "name_case": name_case,
            "detail_case": detail_case,
            "names": names,
            "detailed": detailed,
            "output": output_text,
            "tokens": tokens,
            "tool_called_result": {
                "evaluatorId": "StrandsEvals.ToolCalled",
                "value": tool_called.score,
                "label": "called" if tool_called.test_pass else "not_called",
                "explanation": tool_called.reason,
                "n_tool_calls": n_calls,
                "n_primary_calls": n_primary,
                "tool_called": tool_called.test_pass,
            },
            "budget_result": {
                "evaluatorId": "StrandsEvals.ToolCallBudget",
                "value": budget_result.score,
                "label": budget_result.label,
                "explanation": budget_result.reason,
                "budget": budget,
                "within_budget": budget_result.test_pass,
            },
        }
        budget_flag = "within" if budget_result.test_pass else "OVER"
        tok = f"{tokens} tokens" if tokens is not None else "tokens n/a"
        print(f"  {name}: tool_called={tool_called.test_pass}, "
              f"{n_calls} tool call(s) vs budget {budget} [{budget_flag}], {tok}")
    return runs


print("Running Strands Evals code-based graders (ToolCalled + ToolCallBudget) on the baseline agent...\n")
local_runs = evaluate_agent(create_warehouse_agent, evaluation_scenarios)
tool_called_results = {n: [r["tool_called_result"], r["budget_result"]] for n, r in local_runs.items()}
print("\nCode-based grading complete.")


### LLM-as-judge graders (local)

`ToolCalled` tells you the agent reached its tool, not whether it answered well or worked efficiently. For that we add three LLM-as-judge graders, all routed through the same `LiteLLMModel`:

- **`OutputEvaluator`** (the outcome judge, [docs](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/output_evaluator/)) reads the agent's final answer and the scenario's `expected_response` and scores **task success** on content alone: did the agent return the right inventory data, with the right product codes, without inventing numbers?
- **`TrajectoryEvaluator`** scores the tool-call path against a rubric. Fed the detailed trajectory (endpoints, OData arguments, `is_error` flags), it names the specific redundant or failed calls, so its reason is a diagnosis you can act on.
- **`ToolSelectionAccuracyEvaluator`** is a call-level judge ("was this call justified?"). It needs a Strands `Session` built from OTEL spans, so we drive it through the `Experiment` and `TracedHandler` harness, which captures spans from a fresh local run.

Together the outcome judge and the trajectory judge answer two different questions about the same run: **was the answer right, and was the path to it efficient?**

> These graders make **live LLM calls**, so this cell is slower than the deterministic grader and scores can vary slightly between runs.


In [ ]:
# LLM-as-judge graders with Strands Evals
from strands_evals.evaluators import (
    OutputEvaluator,
    TrajectoryEvaluator,
    ToolSelectionAccuracyEvaluator,
)

# Rubric for the OUTPUT judge: reads the final answer and the scenario's expected_response and
# scores task success on content alone, ignoring the path taken.
OUTPUT_RUBRIC = (
    "You are grading a warehouse inventory agent's final answer for TASK SUCCESS. Compare the "
    "agent's output against the expected response, judging factual content only, ignore wording, "
    "formatting, and style. A successful answer contains the specific inventory data the question "
    "asked for (stock quantities, correct product codes such as WM-AN02, a clear yes/no where a "
    "decision was requested) and does not invent numbers.\n\n"
    "Score 1.0 if the answer fully satisfies the expected response, 0.5 if it is partially correct "
    "or missing key data, and 0.0 if it is wrong, empty, or hallucinated. State briefly in your "
    "reason which required facts are present or missing."
)

# Rubric for the TRAJECTORY judge. We ask the judge to NAME the specific redundant calls it sees,
# so its `reason` becomes an actionable diagnosis rather than just a score.
TRAJECTORY_RUBRIC = (
    "Score the tool-call trajectory for a warehouse inventory agent. Each entry shows the tool "
    "name and the OData arguments used ($filter, $orderby, endpoint, etc.); entries with "
    "`is_error: true` are calls that FAILED. A good trajectory answers the user's question with "
    "as few tool calls as possible, ideally one filtered or sorted OData query.\n\n"
    "Penalize redundant work and, in your reason, IDENTIFY THE SPECIFIC WASTEFUL CALLS: "
    "repeated `$metadata` schema rediscovery, trial-and-error `$filter` guessing (a failed call "
    "followed by retries that differ only in a fumbled filter, the `is_error` flags mark these), "
    "or fetching everything and filtering client-side when a server-side query would do.\n\n"
    "Score 1.0 for an efficient, well-chosen trajectory; lower toward 0.0 as redundant or failed "
    "calls increase. Your reason must explain WHICH calls were wasteful and WHY, and name the "
    "concrete fix (e.g. 'the second and third calls re-fetched $metadata; put the field names in "
    "the system prompt so the agent filters on the first call')."
)

llm_judge_results = {}

print("Running Strands Evals LLM-as-judge graders (via SAP GenAI Hub)...\n")

# OutputEvaluator: the task-success (outcome) judge. include_inputs=True gives the judge the
# original question for context.
output_judge = OutputEvaluator(rubric=OUTPUT_RUBRIC, model=model, include_inputs=True)

for scenario_name, run in local_runs.items():
    try:
        out = output_judge.evaluate(run["name_case"])[0]
        llm_judge_results[scenario_name] = [{
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        }]
        print(f"  {scenario_name}: OutputEvaluator={out.score:.2f} ({'success' if out.test_pass else 'fail'})")
    except Exception as e:
        print(f"  {scenario_name}: OutputEvaluator ERROR: {e}")
        llm_judge_results[scenario_name] = []

print()

# TrajectoryEvaluator: fed the DETAILED trajectory (call args), so it can name the waste.
# `trajectory_description` tells the judge what each tool does (from extract_tools_description).
trajectory_judge = TrajectoryEvaluator(
    rubric=TRAJECTORY_RUBRIC,
    model=model,
    trajectory_description=TOOL_DESCRIPTIONS or None,
)

for scenario_name, run in local_runs.items():
    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        llm_judge_results.setdefault(scenario_name, []).append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        # Print the full reason: this is the "why is it inefficient" diagnosis we want to read.
        print(f"  {scenario_name}: TrajectoryEvaluator={out.score:.2f}")
        print(f"    {out.reason}\n")
    except Exception as e:
        print(f"  {scenario_name}: TrajectoryEvaluator ERROR: {e}\n")

# ToolSelectionAccuracyEvaluator: tool-level judge that needs a Session (OTEL spans). We use the
# Strands Evals Experiment + TracedHandler harness, which runs the agent and captures spans into
# a Session the judge can parse. This is heavier (re-runs the agent) and depends on local
# telemetry capture, so we guard it and fall back gracefully.
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    cases = [
        Case(
            name=s["name"],
            input=s["prompt"],
            expected_trajectory=s["expected_trajectory"],
        )
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def warehouse_eval_task():
        return create_warehouse_agent()

    experiment = Experiment(
        cases=cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    reports = await experiment.run_evaluations_async(warehouse_eval_task, max_workers=1)

    # Each EvaluationReport carries parallel lists: cases[i] (a dict) and scores[i].
    # Fold the tool-selection scores into llm_judge_results, keyed by scenario name.
    for report in reports:
        for case_dict, score in zip(report.cases, report.scores):
            name = case_dict.get("name") if isinstance(case_dict, dict) else None
            if name is not None and score is not None:
                llm_judge_results.setdefault(name, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {name}: ToolSelectionAccuracy={score:.2f}")
    print("\nLLM-as-judge grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracyEvaluator skipped (local trace capture unavailable): {e}")
    print("  OutputEvaluator and TrajectoryEvaluator results above still stand.")

### Local results

We combine the code-based and LLM-judge scores (plus token counts) into one table. The signal to
look for: a scenario that **reached an answer but scored low on the trajectory judge**, correct
but inefficient, exactly what an outcome-only grader misses. For any flagged scenario we print the
judge's explanation and the detailed trajectory so you can see the redundant calls.


In [ ]:
# Combine local Strands Evals scores (code-based + LLM-judge) into one table.
local_results = {}
for scenario in evaluation_scenarios:
    name = scenario["name"]
    local_results[name] = (
        tool_called_results.get(name, [])
        + llm_judge_results.get(name, [])
    )

LOCAL_EVALUATOR_IDS = [
    "StrandsEvals.ToolCalled",
    "StrandsEvals.ToolCallBudget",
    "StrandsEvals.OutputEvaluator",
    "StrandsEvals.TrajectoryEvaluator",
    "StrandsEvals.ToolSelectionAccuracy",
]

# TrajJudge score below this counts as an inefficient trajectory for the LLM-judge diagnosis.
TRAJECTORY_PASS_THRESHOLD = 0.5


def _local_short_name(eid):
    return {
        "StrandsEvals.ToolCalled": "ToolCalled",
        "StrandsEvals.ToolCallBudget": "Budget",
        "StrandsEvals.OutputEvaluator": "TaskSuccess",
        "StrandsEvals.TrajectoryEvaluator": "TrajJudge",
        "StrandsEvals.ToolSelectionAccuracy": "ToolSelect",
    }.get(eid, eid.split(".")[-1][:12])


# Tokens is an informational efficiency signal (not a pass/fail grade): fewer tokens for the
# same correct answer is a cheaper, tighter trajectory.
headers = ["Scenario"] + [_local_short_name(eid) for eid in LOCAL_EVALUATOR_IDS] + ["Tokens"]
rows = []
local_scenario_scores = defaultdict(dict)
for name, results in local_results.items():
    cells = [name]
    for eid in LOCAL_EVALUATOR_IDS:
        score = next((r.get("value", "-") for r in results if r.get("evaluatorId") == eid), "-")
        if isinstance(score, (int, float)):
            cells.append(f"{score:.2f}")
            local_scenario_scores[name][eid] = score
        else:
            cells.append(str(score))
    tokens = local_runs.get(name, {}).get("tokens")
    cells.append(tokens if isinstance(tokens, int) else "-")
    rows.append(cells)

md_table(headers, rows, title="Local evaluation — Strands Evals (code-based + LLM-judge)")

# Highlight correct-but-inefficient: the task succeeded (TaskSuccess high) but the agent still
# blew its tool-call budget. We gate on the DETERMINISTIC ToolCallBudget grader, not the LLM
# trajectory score, so this flag is reproducible run to run: a right answer reached the wrong way
# is caught every time, not only when the judge happens to score the path low. The LLM judge's
# reason is rendered alongside as the qualitative "why" (which specific calls were wasteful).
diagnosis = ["#### Correct-but-inefficient check (task succeeded, but OVER the tool-call budget)\n"]
flagged = False
for name, results in local_results.items():
    output = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.OutputEvaluator"), {})
    budget = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.ToolCallBudget"), {})
    traj = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.TrajectoryEvaluator"), {})
    output_score = output.get("value")
    # "Succeeded" means the OUTCOME judge actually passed the answer — a tool merely being called
    # is not success (that conflated a wrong-but-tool-using answer with a correct one).
    succeeded = isinstance(output_score, (int, float)) and output_score >= 0.5
    over_budget = budget.get("within_budget") is False
    if succeeded and over_budget:
        flagged = True
        called = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.ToolCalled"), {})
        n_calls = called.get("n_tool_calls", "?")
        tokens = local_runs.get(name, {}).get("tokens")
        traj_score = traj.get("value")
        traj_str = f"{traj_score:.2f}" if isinstance(traj_score, (int, float)) else "n/a"
        diagnosis.append(
            f"**⚠️ {name}**: succeeded (TaskSuccess {output_score:.2f}) but used {n_calls} tool "
            f"call(s) / {tokens} tokens — OVER budget ({budget.get('explanation', '')}). "
            f"TrajJudge scored {traj_str}.\n"
        )
        diagnosis.append(f"- **Why:** {traj.get('explanation', '')}")
        diagnosis.append("- **Trajectory:**")
        for i, call in enumerate(local_runs.get(name, {}).get("detailed", []), 1):
            diagnosis.append(f"    {i}. `{call}`")
        diagnosis.append("")
if not flagged:
    diagnosis.append("None flagged: every successful answer stayed within its tool-call budget.")
show_md("\n".join(diagnosis))


### Apply a fix and re-test

The trajectory judge did not just score the baseline low, it named *why*: an unnecessary selector
round-trip, repeated `$metadata` rediscovery, and trial-and-error `$filter` guessing, and it
recommended putting the schema where the query is built so the agent answers in one call. That is a
diagnosis of the **architecture**, not just the prompt: all three wasted steps rediscover, at
runtime, facts that are fixed at design time.

So the fix is architectural. `util/warehouse_agent.py` ships `build_improved_warehouse_agent`, which
adds a deep **`get_warehouse_stock`** tool: the SAP service root, entity set, field names, warehouse
ID, and server-side `$filter`/`$select` all live *in code*, so a stock question becomes a
single, correct-by-construction call, no selector, no `$metadata`, no fumbled filters. The selector
and generic `odata_caller` stay wired in as a **fallback** for genuinely off-path questions, so the
improved agent is a *router*: deep tool on the hot path, dynamic discovery for everything else. This
maps each judge finding to a concrete code change:

| Judge finding (baseline) | Fix in `get_warehouse_stock` |
|---|---|
| unnecessary selector round-trip | no selector on the stock path; the router calls the tool directly |
| repeated `$metadata` rediscovery | field names compiled into `$select` |
| trial-and-error `$filter` guessing | `$filter` built deterministically in Python |

We re-run the same scenarios against the improved agent and grade it with the **same five Strands
Evals graders** (ToolCalled, Budget, TaskSuccess, TrajJudge, ToolSelect), now counting its primary tool,
`get_warehouse_stock`. The next cell prints an overview on the same axes as the baseline, then a
**before → after** table where the trajectory collapses from `odata_caller` × N (with selector +
`$metadata`) to `get_warehouse_stock` × 1. This closes the loop: measure, diagnose, fix, re-measure.

In [ ]:
# Re-run the scenarios against the IMPROVED agent (deep get_warehouse_stock tool + fallback), then
# grade it with the SAME five Strands Evals graders used on the baseline (code-based ToolCalled +
# ToolCallBudget + the three LLM judges), so the fix is measured on every metric, not just the
# trajectory score. We pass IMPROVED_TOOL_NAME so ToolCalled and the expected trajectory target the
# deep tool the improved agent actually uses. Results are stored in `fixed_results`, mirroring
# `local_results`, so the next cell can print a full overview.
print("Re-running scenarios against the improved agent (deep-tool architecture)...\n")

# evaluate_agent runs each scenario once and applies the code-based graders (ToolCalled +
# ToolCallBudget), counting get_warehouse_stock (the improved agent's primary tool) not odata_caller.
fixed_runs = evaluate_agent(create_improved_warehouse_agent, evaluation_scenarios,
                            tool_name=IMPROVED_TOOL_NAME)

# Grade the fixed runs with the same LLM judges (OutputEvaluator, TrajectoryEvaluator) reused
# from the baseline cell, building per-scenario result lists shaped exactly like `local_results`.
print("\nGrading fixed answers and trajectories with the LLM judges...\n")
fixed_results = {}
for name, run in fixed_runs.items():
    # Code-based graders (deterministic): ToolCalled + ToolCallBudget.
    results = [run["tool_called_result"], run["budget_result"]]

    # OutputEvaluator is the TaskSuccess (outcome) judge; keep its score so we can print it
    # alongside TrajJudge below and the two labels aren't conflated.
    task_success = None
    try:
        out = output_judge.evaluate(run["name_case"])[0]
        task_success = out.score
        results.append({
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
    except Exception as e:
        print(f"  {name}: OutputEvaluator ERROR: {e}")

    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        run["traj_score"] = out.score
        run["traj_reason"] = out.reason
        results.append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        # Print BOTH scores, correctly labeled: TaskSuccess is the OutputEvaluator (did it answer
        # right?), TrajJudge is the TrajectoryEvaluator (did it get there efficiently?).
        ts = f"{task_success:.2f}" if isinstance(task_success, (int, float)) else "n/a"
        print(f"  {name}: TaskSuccess={ts}  TrajJudge={out.score:.2f}")
    except Exception as e:
        run["traj_score"] = None
        run["traj_reason"] = f"error: {e}"
        print(f"  {name}: TrajectoryEvaluator ERROR: {e}")

    fixed_results[name] = results

# ToolSelectionAccuracy needs OTEL spans, so it runs through the same Experiment + TracedHandler
# harness used on the baseline, here with the improved (deep-tool) agent. Its expected_trajectory
# targets get_warehouse_stock. Guarded and optional.
print("\nGrading fixed tool selection (needs local trace capture)...\n")
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    fixed_cases = [
        Case(name=s["name"], input=s["prompt"], expected_trajectory=[IMPROVED_TOOL_NAME])
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def fixed_warehouse_eval_task():
        return create_improved_warehouse_agent()

    fixed_experiment = Experiment(
        cases=fixed_cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    fixed_reports = await fixed_experiment.run_evaluations_async(fixed_warehouse_eval_task, max_workers=1)

    for report in fixed_reports:
        for case_dict, score in zip(report.cases, report.scores):
            cname = case_dict.get("name") if isinstance(case_dict, dict) else None
            if cname is not None and score is not None:
                fixed_results.setdefault(cname, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {cname}: ToolSelectionAccuracy={score:.2f}")
    print("\nFixed-agent grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracy skipped for fixed agent (local trace capture unavailable): {e}")
    print("  ToolCalled, TaskSuccess and TrajJudge results above still stand.")

In [ ]:
# Fixed-agent overview + full before/after comparison, across ALL local graders (not just the
# trajectory judge). The first table mirrors the baseline "Local evaluation" table above so you
# can read the improved agent on the same axes; the second puts every metric side by side,
# before/after. Note the Calls column: the primary tool changes from odata_caller (baseline) to
# get_warehouse_stock (improved), so this counts N wasteful OData calls collapsing to 1 deep call.


def _score_for(results, eid):
    """Pull one evaluator's score out of a per-scenario result list (or None)."""
    return next((r.get("value") for r in results if r.get("evaluatorId") == eid), None)


def _fmt(score):
    return f"{score:.2f}" if isinstance(score, (int, float)) else "-"


# --- Fixed-agent overview (same columns as the baseline local-evaluation table) ---
headers = ["Scenario"] + [_local_short_name(eid) for eid in LOCAL_EVALUATOR_IDS] + ["Tokens"]
rows = []
for name in (s["name"] for s in evaluation_scenarios):
    cells = [name] + [_fmt(_score_for(fixed_results.get(name, []), eid)) for eid in LOCAL_EVALUATOR_IDS]
    tokens = fixed_runs.get(name, {}).get("tokens")
    cells.append(tokens if isinstance(tokens, int) else "-")
    rows.append(cells)

md_table(headers, rows,
         title="Improved agent (deep get_warehouse_stock tool) — Strands Evals (code-based + LLM-judge)")

# --- Before/after, every metric (tool calls, tokens, and all four graders) ---
ba_headers = ["Scenario", "Calls b→a", "Tokens b→a"] + [
    _local_short_name(eid) + " b→a" for eid in LOCAL_EVALUATOR_IDS
]
ba_rows = []
for name in (s["name"] for s in evaluation_scenarios):
    base, fix = local_runs.get(name, {}), fixed_runs.get(name, {})
    base_calls = base.get("tool_called_result", {}).get("n_tool_calls", "?")
    fix_calls = fix.get("tool_called_result", {}).get("n_tool_calls", "?")
    base_tok, fix_tok = base.get("tokens", "-"), fix.get("tokens", "-")
    cells = [name, f"{base_calls}→{fix_calls}", f"{base_tok}→{fix_tok}"]
    for eid in LOCAL_EVALUATOR_IDS:
        b = _fmt(_score_for(local_results.get(name, []), eid))
        a = _fmt(_score_for(fixed_results.get(name, []), eid))
        cells.append(f"{b}→{a}")
    ba_rows.append(cells)

reading = (
    "**Reading it:** the *Calls* column tells the story — a multi-call `odata_caller` trajectory "
    "(selector + `$metadata` + trial-and-error `$filter`) collapses to a single `get_warehouse_stock` "
    "call, so TrajJudge jumps and tokens drop, while TaskSuccess stays high. The deep tool made the "
    "agent faster and more reliable, not just better-prompted. That is the whole point of grading "
    "trajectory alongside outcome, and of fixing the architecture the judge pointed at."
)
md_table(ba_headers, ba_rows,
         title="Before → after (baseline vs. improved agent) — all metrics", note=reading)


---

## Stage 2: Deployed evaluation with AgentCore

Stage 1 answered *"is the logic good?"* and produced a fix. But that fix only lives in this
notebook. Stage 2 answers the second question: **did the fix ship, and how does the real thing
behave under production infrastructure?**

So the flow here is deploy-then-verify:

1. **Deploy the improved agent** as a new A2A-protocol AgentCore Runtime, secured with the
   same Cognito JWT authorizer from Lab 7.
2. **Grade the improved runtime** with **AgentCore Evaluations**, which scores it from its
   **OTEL traces**: the deployed agent emits spans to CloudWatch, we invoke it per scenario
   via JSON-RPC 2.0 and wait (~180s) for ingestion, then `EvaluationClient.run()` reads
   those spans and scores them with built-in and custom LLM-as-judge evaluators.
3. **Render the full outcome overview**: every evaluator's grade for every scenario
   (Correctness / Helpfulness / GoalSuccessRate / ToolSelectionAccuracy / OpsQuality) plus
   prod-only latency, framed as a before/after against Stage 1.

This is what you run against the *actual* production agent, and the same mechanism you would
wire into continuous online evaluation for monitoring.

### Configure evaluation infrastructure

Set up the A2A invocation helpers and the OTEL ingestion wait.


In [ ]:
# Condition: CloudWatch Transaction Search must be ON for Stage 2 to have traces to grade.
xray_client = boto3.client("xray", region_name=REGION)

try:
    dest = xray_client.get_trace_segment_destination()
    destination = dest.get("Destination")
    status = dest.get("Status")
    if destination == "CloudWatchLogs" and status == "ACTIVE":
        print(f"Transaction Search is enabled (destination={destination}, status={status}).")
        print("Stage 2 will be able to read the deployed agent's traces.")
    else:
        raise SystemExit(
            "CloudWatch Transaction Search is not fully enabled "
            f"(destination={destination}, status={status}). Stage 2 needs it to read the "
            "deployed agent's OTEL spans as traces.\n\n"
            "Enable it once per account/region:\n"
            "  CloudWatch console -> Application Signals -> Transaction Search -> Enable,\n"
            "  or: aws xray update-trace-segment-destination --destination CloudWatchLogs\n"
            "     aws xray update-indexing-rule ...  (see the Transaction Search setup docs)\n\n"
            "Docs: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html"
        )
except SystemExit:
    raise
except Exception as e:
    # A permissions or API error here shouldn't hard-block Stage 2 (the account may still be
    # configured); surface it as a warning so the user can decide whether to proceed.
    print(f"WARNING: could not verify Transaction Search status ({e}).")
    print("If Stage 2 grading returns empty scores, confirm Transaction Search is enabled for this account/region.")

In [ ]:
# A2A invocation helpers -- call the deployed agent via JSON-RPC 2.0 over HTTPS.
# The Cognito bearer token (BEARER_TOKEN env var, set in the deploy cell) authenticates each call.
import requests as _requests
import uuid as _uuid
from urllib.parse import quote as _quote

# Time to wait for OTEL spans to arrive in CloudWatch after invocation.
INGESTION_DELAY = 180

def _build_invoke_url(agent_arn: str, region: str) -> str:
    """Build the AgentCore A2A invoke URL for a given agent ARN."""
    encoded = _quote(agent_arn, safe="")
    return f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded}/invocations?qualifier=DEFAULT"

def _make_jsonrpc_payload(text: str, context_id: str) -> dict:
    """Build a JSON-RPC 2.0 message/send payload with a stable context_id."""
    return {
        "jsonrpc": "2.0",
        "id": str(_uuid.uuid4()),
        "method": "message/send",
        "params": {
            "contextId": context_id,
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
                "messageId": str(_uuid.uuid4()),
            },
        },
    }

def _extract_a2a_text(rpc_response: dict) -> str:
    """Concatenate text parts from all artifacts in an A2A JSON-RPC response."""
    artifacts = rpc_response.get("result", {}).get("artifacts", [])
    return "".join(
        part["text"]
        for artifact in artifacts
        for part in artifact.get("parts", [])
        if part.get("kind") == "text" and part.get("text")
    ).strip()

def invoke_a2a(invoke_url: str, bearer_token: str, prompt: str, context_id: str,
               max_retries: int = 3, wait: int = 30):
    """POST a JSON-RPC 2.0 message/send to the A2A endpoint, retrying cold-start failures.

    Returns (agent_text, effective_context_id, cold_start_flag). On cold start the
    context_id gains a '_r{n}' suffix on each retry so every attempt is a unique session.
    grade_sessions must use the returned effective_context_id to find the right OTEL spans.
    """
    import time as _time
    effective_id = context_id
    for attempt in range(max_retries):
        try:
            resp = _requests.post(
                invoke_url,
                headers={"Authorization": f"Bearer {bearer_token}",
                         "Content-Type": "application/json"},
                json=_make_jsonrpc_payload(prompt, effective_id),
                timeout=120,
            )
            if resp.status_code == 200:
                text = _extract_a2a_text(resp.json())
                return text, effective_id, attempt > 0
            if resp.status_code >= 500 and attempt < max_retries - 1:
                print(f"  HTTP {resp.status_code} (attempt {attempt+1}/{max_retries}), "
                      f"retrying in {wait}s (likely cold start)...")
                _time.sleep(wait)
                effective_id = f"{context_id}_r{attempt+1}"
            else:
                raise RuntimeError(f"A2A invoke failed: HTTP {resp.status_code} -- {resp.text[:300]}")
        except (_requests.ConnectionError, _requests.Timeout) as e:
            if attempt < max_retries - 1:
                print(f"  Connection error (attempt {attempt+1}/{max_retries}): {e} -- retrying...")
                _time.sleep(wait)
                effective_id = f"{context_id}_r{attempt+1}"
            else:
                raise
    raise RuntimeError("All A2A invoke attempts failed.")

def invoke_with_timing(invoke_url: str, bearer_token: str, prompt: str, context_id: str):
    """Wrap invoke_a2a with wall-clock latency. Returns (agent_text, timing_dict)."""
    import time as _time
    start = _time.time()
    agent_text, effective_id, cold = invoke_a2a(invoke_url, bearer_token, prompt, context_id)
    return agent_text, {
        "latency_s": _time.time() - start,
        "cold_start": cold,
        "context_id": effective_id,
    }

print("A2A invocation helpers ready.")
print(f"CloudWatch ingestion delay: {INGESTION_DELAY}s")

### Create the custom evaluator

We register a domain-specific LLM-as-a-judge evaluator in the AgentCore control plane. This complements the built-in evaluators with SAP warehouse-specific scoring criteria that generic evaluators cannot assess.

**WarehouseOperationalQuality** (TRACE-level): evaluates whether the agent's response is operationally useful for a warehouse manager: does it provide actionable inventory insights, use correct product codes, and present data in a way that supports procurement decisions?

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

_SUFFIX = uuid.uuid4().hex[:8]

# TODO: Use the inference profile matching your region (us.* for us-east-1, eu.* for eu-central-1)
JUDGE_MODEL_ID = "us.amazon.nova-pro-v1:0"

# Custom TRACE-level evaluator: Warehouse Operational Quality
# NOTE: `instructions` carries only the rubric (what to judge and when each score applies). We
# deliberately do NOT hand-write output-format directives ("respond with the score on the first
# line", etc.): the service derives the response format from the `ratingScale` below and appends
# its own formatting scaffolding to the prompt. A hand-written format block would duplicate — and
# risk contradicting — that, which can make the judge's output unparseable and silently drop scores.
print("Creating WarehouseOperationalQuality evaluator (TRACE-level)...")
warehouse_quality_response = agentcore_control.create_evaluator(
    evaluatorName=f"WarehouseOperationalQuality_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "You are a warehouse operations expert evaluating an AI assistant that queries "
                "SAP S/4HANA warehouse APIs for inventory management.\n\n"
                "Conversation context: {context}\n"
                "Agent response: {assistant_turn}\n"
                "Expected behavior: {expected_response}\n\n"
                "Evaluate the OPERATIONAL QUALITY of the response for a warehouse manager. Score based on:\n"
                "1. Does the response contain specific, quantitative inventory data (not vague statements)?\n"
                "2. Does it use the SAP product codes that appear in the API results correctly, and "
                "attach quantities to the right product (rather than inventing codes or mislabeling)?\n"
                "3. Is the data presented in a way that supports immediate operational decisions "
                "(e.g., reorder recommendations, fulfillment feasibility, capacity utilization)?\n"
                "4. Does the response avoid hallucinating inventory numbers or product codes when API "
                "data is unavailable?\n\n"
                "Important: If the agent successfully queried the API and returned real data with the "
                "correct product codes for THIS query and actionable insights, score 1.0 even if the "
                "specific products or formatting differ from the expected response. Do not require any "
                "particular product code to be present - judge only that the codes actually returned "
                "are used correctly."
            ),
            "ratingScale": {
                "numerical": [
                    {"value": 0.0, "label": "not_actionable", "definition": "Response lacks data, hallucinates numbers, or provides no operational value."},
                    {"value": 0.5, "label": "partially_useful", "definition": "Some useful data present but missing key operational context for decisions."},
                    {"value": 1.0, "label": "operationally_excellent", "definition": "Accurate, specific, and actionable warehouse intelligence."},
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": JUDGE_MODEL_ID,
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_EVALUATOR_ID = warehouse_quality_response["evaluatorId"]
print(f"  Created: {CUSTOM_EVALUATOR_ID}")
print(f"\nCustom evaluator registered in AgentCore control plane.")

### Ship the Stage-1 fix: deploy the improved agent as a new A2A runtime

Stage 1 found the baseline agent correct-but-inefficient and produced a fix, the deep-tool
architecture (`build_improved_warehouse_agent`), but that fix only lives in this notebook.
Before we can verify it in production, we have to ship it. The next cell deploys the improved
agent as a **new A2A-protocol AgentCore Runtime**, secured with the same Cognito JWT authorizer
from Lab 7. `deploy_improved_agent()` (in `util/deploy_agentcore.py`) handles all the
plumbing: entrypoint, Dockerfile, configure, launch, and READY poll.

> **Time and cost:** the deploy cell triggers a real CodeBuild and takes about 4 min; the full
> Stage 2 (deploy, invoke per scenario, ~180s OTEL ingestion wait, grade) runs about 9-12 min
> total. It also makes live Bedrock/AgentCore calls, so it incurs cost.

In [ ]:
# Deploy the improved agent as a new A2A-protocol AgentCore Runtime.
# deploy_improved_agent() (util/deploy_agentcore.py) writes the StrandsA2AExecutor entrypoint,
# patches the Dockerfile, configures with protocol="A2A" + Cognito JWT authorizer, launches,
# and blocks until READY. Returns the launch_result object (agent_id, agent_arn).
from util.deploy_agentcore import deploy_improved_agent

bearer_token = os.environ.get("BEARER_TOKEN", "")
if not bearer_token:
    import getpass
    bearer_token = getpass.getpass("BEARER_TOKEN (from Lab 7 Step 6c): ")
    os.environ["BEARER_TOKEN"] = bearer_token

lab9_launch = deploy_improved_agent(
    agent_name="warehouse_ops_agent_eval_improved",
    region=REGION,
    sap_api_key=os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"],
    discovery_url=DISCOVERY_URL,
    client_id=CLIENT_ID,
    model_id=model.model_id,
)

# Store ARN/ID so downstream cells build the correct A2A invoke URL and query the right traces.
AGENT_ARN = lab9_launch.agent_arn
AGENT_ID  = lab9_launch.agent_id
os.environ["LAB9_AGENT_ARN"] = AGENT_ARN
os.environ["LAB9_AGENT_ID"]  = AGENT_ID

A2A_INVOKE_URL = _build_invoke_url(AGENT_ARN, REGION)
print(f"Improved agent deployed and ready for Stage 2 evaluation.")
print(f"  Agent ID  : {AGENT_ID}")
print(f"  Agent ARN : {AGENT_ARN}")
print(f"  Invoke URL: {A2A_INVOKE_URL}")

### Invoke the deployed agent for each scenario

We invoke the redeployed (improved) agent for each evaluation scenario. Each invocation gets a unique `runtimeSessionId` so the evaluator can locate its spans independently.

In [ ]:
def run_prod_round(label, scenarios):
    """Invoke each scenario against the deployed A2A runtime via JSON-RPC 2.0, wait for
    span ingestion, and return per-scenario session dicts (timing + effective context_id).

    The effective context_id (which may carry a '_r{n}' suffix after a cold-start retry) is
    stored as 'session_id' -- grade_sessions uses it to query the right OTEL spans.
    """
    _bearer = os.environ.get("BEARER_TOKEN", "")
    if not _bearer:
        raise SystemExit("BEARER_TOKEN not set -- run the deploy cell (Step 9) first.")
    if not AGENT_ARN:
        raise SystemExit("AGENT_ARN not set -- run the deploy cell (Step 9) first.")

    invoke_url = _build_invoke_url(AGENT_ARN, REGION)
    sessions = []
    print(f"[{label}] Invoking deployed A2A agent for {len(scenarios)} scenarios...\n")

    for scenario in scenarios:
        context_id = f"eval_{scenario['name']}_{_uuid.uuid4().hex}"
        try:
            agent_text, timing = invoke_with_timing(invoke_url, _bearer, scenario["prompt"], context_id)
            sessions.append({
                "scenario_name": scenario["name"],
                # grade_sessions queries OTEL spans by 'session_id'; for A2A that is the
                # effective context_id the successful invocation ran under.
                "session_id": timing["context_id"],
                "prompt": scenario["prompt"],
                "response": agent_text[:500],
                "full_response": agent_text,
                "expected_response": scenario["expected_response"],
                "expected_trajectory": [IMPROVED_TOOL_NAME],
                "assertions": scenario["assertions"],
                "latency_s": timing["latency_s"],
                "cold_start": timing["cold_start"],
            })
            cold = " (cold start)" if timing["cold_start"] else ""
            show_md(
                f"**Prompt:** {scenario['prompt']}\n\n"
                f"_Latency: {timing['latency_s']:.1f}s{cold}_\n\n{agent_text}",
                title=scenario["name"],
            )
        except Exception as e:
            print(f"  [{scenario['name']}] ERROR: {e}")

    print(f"\n[{label}] Waiting {INGESTION_DELAY}s for CloudWatch span ingestion...")
    time.sleep(INGESTION_DELAY)
    return sessions

### Evaluate with built-in evaluators

We use `EvaluationClient.run()` to score each session with AgentCore's built-in evaluators:

| Evaluator | Level | Needs Ground Truth | What it measures |
|-----------|-------|-------------------|-----------------|
| `Builtin.Correctness` | TRACE | `expected_response` | Factual accuracy of the response |
| `Builtin.Helpfulness` | TRACE | None | How useful/valuable the response is |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` | Whether the agent completed the user's goal |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | None | Whether each individual tool call was justified |

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

ec = EvaluationClient(region_name=REGION)

# Pre-populate the evaluator-level cache. This is the documented setup for Builtin.* evaluators:
# EvaluationClient.run() resolves each evaluator's level (SESSION / TRACE / TOOL_CALL) to build the
# right evaluationTarget, but its get_evaluator lookup does NOT return a level for built-in IDs, so
# on that path the SDK silently falls back to SESSION — wrong for TRACE and TOOL_CALL evaluators.
# Pre-seeding the cache is what prevents that mis-classification, and it's exactly what the official
# aws-samples example does (amazon-bedrock-agentcore-samples,
# 01-features/06-observe-evaluate-optimize-your-agent). Levels are from the built-in prompt-templates
# docs: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/prompt-templates-builtin.html
# NOTE: ToolSelectionAccuracy is a TOOL_CALL (tool-level) evaluator, not SESSION — the docs classify
# it under "Tool-level evaluators" (it judges whether each individual tool call was justified).
ec._evaluator_level_cache.update({
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.ToolSelectionAccuracy": "TOOL_CALL",
})

BUILTIN_EVALUATOR_IDS = [
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.GoalSuccessRate",
    "Builtin.ToolSelectionAccuracy",
]

# The custom WarehouseOperationalQuality evaluator (registered above) is TRACE-level too, so
# grade_sessions can run it in the same pass as the built-ins.
ec._evaluator_level_cache[CUSTOM_EVALUATOR_ID] = "TRACE"


def grade_sessions(sessions):
    """Grade prod sessions with the built-in + custom AgentCore evaluators.

    Runs BUILTIN_EVALUATOR_IDS then the custom WarehouseOperationalQuality evaluator on each
    session's OTEL traces and returns {scenario_name: [built-in results..., custom results...]}
    the combined shape render_overview consumes. Any evaluator error falls back to
    [] for that group so one bad session never aborts the round.
    """
    graded = {}
    print(f"Grading {len(sessions)} session(s) with AgentCore built-in + custom evaluators...\n")
    for session in sessions:
        name = session["scenario_name"]
        print(f"  Evaluating: {name} (session: {session['session_id']})")

        reference_inputs = ReferenceInputs(
            expected_response=session["expected_response"],
            expected_trajectory=session["expected_trajectory"],
            assertions=[session["assertions"]],
        )

        try:
            builtin = ec.run(
                evaluator_ids=BUILTIN_EVALUATOR_IDS,
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in builtin:
                print(f"    {r.get('evaluatorId', 'unknown')}: {r.get('value', 'N/A')} ({r.get('label', '')})")
        except Exception as e:
            print(f"    built-in ERROR: {e}")
            builtin = []

        try:
            custom = ec.run(
                evaluator_ids=[CUSTOM_EVALUATOR_ID],
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in custom:
                label = r.get("label", "")
                print(f"    OpsQuality: {r.get('value', 'N/A')} ({label})")
                if r.get("explanation"):
                    print(f"      {r['explanation'][:120]}")
        except Exception as e:
            print(f"    custom ERROR: {e}")
            custom = []

        graded[name] = builtin + custom
        print()

    print("Grading complete.")
    return graded


### Grade the improved runtime

Now we run the whole prod round end to end: `run_prod_round` invokes the deployed (improved,
redeployed) agent for every scenario, and `grade_sessions` scores each one with the four built-in
evaluators plus the custom **WarehouseOperationalQuality** judge. The custom judge scores whether
the agent provides actionable warehouse intelligence, something the generic built-ins cannot assess.

> `run_prod_round` does a live AgentCore invocation per scenario and then waits ~180s for OTEL spans
> to land in CloudWatch, so this cell is slow. Run it once.

In [ ]:
# Run the full prod round: invoke the deployed (improved) runtime and grade every session.
# run_prod_round does live invocations + a ~180s ingestion wait, so this is the slow cell — run it once.
improved_sessions = run_prod_round("improved", evaluation_scenarios)
improved_graded = grade_sessions(improved_sessions)

### Stage 2 outcome overview

`grade_sessions` produced five evaluator grades for every scenario. `render_overview` lays out the
complete result in one table: one row per scenario, one column per evaluator, plus prod-only latency
(cold starts flagged with `*`), closing with a mean row. 

In [ ]:
from util.deployed_eval import render_overview

# One column per evaluator grade_sessions emits, in report order. render_overview shows every
# scenario's grade for every evaluator (nothing averaged away) plus prod-only latency and a mean row.
OVERVIEW_COLUMNS = [
    ("Correct", "Builtin.Correctness"),
    ("Helpful", "Builtin.Helpfulness"),
    ("GoalSuc", "Builtin.GoalSuccessRate"),
    ("ToolSel", "Builtin.ToolSelectionAccuracy"),
    ("OpsQual", CUSTOM_EVALUATOR_ID),
]

# render_overview returns a Markdown table; show_md renders it as a rich HTML table (like Lab 7).
show_md(render_overview(improved_sessions, improved_graded, OVERVIEW_COLUMNS),
        title="Stage 2: improved runtime (prod, redeployed)")


## Cleanup (optional)

In [ ]:
# Uncomment to clean up the custom evaluator created in this lab.
# The deployed runtime (warehouse_ops_agent_eval_improved) is owned by this lab (Lab 9).
# Uncomment below to delete it when done with evaluation.

# agentcore_control.delete_evaluator(evaluatorId=CUSTOM_EVALUATOR_ID)
# print(f"Deleted evaluator: {CUSTOM_EVALUATOR_ID}")

## Summary

You evaluated the warehouse agent in **two stages**, grounded in Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

**Grader taxonomy:** code-based (fast, objective), model-based / LLM-as-judge (flexible, nuanced),
human (gold standard, for calibration), plus the outcome-vs-trajectory distinction.

**Stage 1: Local evaluation with Strands Evals** (fast, cheap pre-deploy gate):
- Code-based `ToolCalled`; LLM-as-judge `TrajectoryEvaluator` (fed each call's OData arguments so it
  *names* the wasteful calls) and `ToolSelectionAccuracyEvaluator`, plus token counts as an
  informational efficiency signal, all routed through the same `LiteLLMModel`.
- Caught a **correct-but-inefficient** answer (the low-stock query) that outcome graders score 1.0,
  then **applied the judge's fix and re-tested**. The judge's diagnosis pointed past the prompt to
  the *architecture*: it was rediscovering, at runtime, facts fixed at design time. So the fix is a
  deep **`get_warehouse_stock`** tool that encodes the schema and query in code (with the selector +
  `odata_caller` kept as a fallback), collapsing an N-call `odata_caller` trajectory to a single
  call while keeping the answer correct, the full measure → diagnose → fix → re-measure loop.

**Stage 2: Deployed evaluation with AgentCore** (deploy-then-verify on the real runtime):
- **Shipped the Stage-1 fix**: deployed the improved deep-tool agent as a new A2A-protocol
  AgentCore Runtime (secured with the Lab 7 Cognito JWT authorizer), because the fix had only
  ever lived locally.
- **Invoked via JSON-RPC 2.0** over HTTPS with a Cognito bearer token -- the same protocol
  an external client (e.g. Joule) would use.
- **Verified in production** from OTEL traces: built-in evaluators (Correctness, Helpfulness,
  GoalSuccessRate, ToolSelectionAccuracy) plus a custom LLM-as-judge evaluator
  (WarehouseOperationalQuality) confirmed quality **held (parity)**, alongside **prod-only latency**,
  the signal a local run physically can't produce.
- The before→after is therefore **cross-stage**: Stage-1 local (naive baseline, flagged inefficient)
  -> Stage-2 prod (improved A2A deep-tool agent, measured where it actually runs).

**Key takeaway:** evaluate locally first, then ship the fix and verify it in production; combine
code-based and model-based graders so you measure not just *whether* the agent is right (outcome)
but *how efficiently* it gets there (trajectory); and when the trajectory judge keeps naming the
same runtime rediscovery, fix the **architecture** it points at (a deep, typed tool), not just the
prompt, then re-grade.

**Where to take this next:** the natural extensions to this harness are negative / out-of-scope
scenarios, multi-trial runs with pass@k / pass^k, chaos / fault-injection (Strands Evals 1.0+), and
a pass/fail regression gate that blocks a deploy when a grader drops below threshold.